In [ ]:
import torch
from pathlib import Path
from neuralhydrology.utils.config import Config
from neuralhydrology.training.train import start_training

# Set the device
if torch.cuda.is_available():
    device = "cuda:0"
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"
    print("Using CPU")

In [ ]:
# Load the configuration
config_path = Path("DE_2_basin_SequentialForecastLSTM.yml")
cfg = Config(config_path)
cfg.device = device

print(f"Loaded config for experiment: {cfg.experiment_name}")
print(f"Run directory: {cfg.run_dir}")

## Handle Zarr Cache

We check if a Zarr cache exists from a previous run to avoid rebuilding the dataset. We also ensure that `train_dir` is set to `None` so that a new run directory is created for this training session, where the scaler will be saved.

In [ ]:
# Capture the location of the Zarr cache if it exists in the config's train_dir
if cfg.train_dir is not None:
    zarr_cache_path = cfg.train_dir / "train_data.zarr"
else:
    zarr_cache_path = None

# Modify config to use the Zarr cache and default run directory structure
update_dict = {
    "train_dir": None,
    "save_train_data": False
}

if zarr_cache_path and zarr_cache_path.exists():
    print(f"Using existing Zarr cache at: {zarr_cache_path}")
    update_dict["train_data_file"] = zarr_cache_path
else:
    print("Warning: Zarr cache not found. Dataset may be rebuilt.")

cfg.update_config(update_dict)

In [ ]:
# Start training
start_training(cfg)